# Lab 9.3: Mixed Workload Management Simulator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/10_production_stories/09.3_mixed_workload_management/lab.ipynb)
[![Open In Molab](https://raw.githubusercontent.com/marimo-team/marimo/main/docs/_static/marimo-badge.svg)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/10_production_stories/09.3_mixed_workload_management/lab.ipynb)

Simulate a shared GPU fleet serving mixed workloads with priority scheduling, preemption, and SLO tracking.

In [ ]:
# Install dependencies (subprocess for Colab/Molab compatibility)
import subprocess, sys
# Install numpy and matplotlib for numerical simulation and visualization
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === PARAMETERS (change these and re-run) ===
# Total number of GPUs in the shared fleet
FLEET_SIZE = 100
# Fraction of fleet reserved exclusively for P0 real-time traffic
RESERVED_FRACTION = 0.1
# Simulation duration in seconds
SIM_DURATION = 3600
# Random seed for reproducibility
SEED = 42

In [ ]:
# Define workload classes with their characteristics
# Each class has: arrival rate (req/s), context length (tokens),
# generation length (tokens), TTFT SLO (ms), priority level
# Define three workload classes with distinct SLO and priority characteristics
WORKLOADS = {
    # P0: ultra-low-latency voice/autocomplete requiring dedicated GPU reservation
    'P0_realtime': {
        'arrival_rate': 50,      # 50 requests per second
        'ctx_len': 512,          # short context for voice/autocomplete
        'gen_len': 32,           # terse output (few tokens)
        'ttft_slo_ms': 200,      # strict latency requirement
        'priority': 0,           # highest priority, never preempted
        'cost_multiplier': 3.0,  # premium pricing for guaranteed SLO  # premium pricing for guaranteed SLO
    },
    # P1: standard chatbot/RAG traffic forming bulk of demand
    'P1_interactive': {
        'arrival_rate': 200,     # bulk of traffic from chatbots/RAG
        'ctx_len': 4096,         # medium context conversations
        'gen_len': 256,          # standard response length
        'ttft_slo_ms': 2000,     # 2 second target
        'priority': 1,           # high priority, rarely preempted
        'cost_multiplier': 1.5,  # moderate premium for priority service  # moderate premium
    },
    # P3: background batch jobs that tolerate preemption for cost savings
    'P3_batch': {
        'arrival_rate': 100,     # background summarization jobs
        'ctx_len': 8192,         # long documents
        'gen_len': 512,          # verbose summaries
        'ttft_slo_ms': 60000,    # 60 second tolerance
        'priority': 3,           # lowest priority, frequently preempted
        'cost_multiplier': 0.4,  # 60% discount for preemptible batch  # 60% discount for preemptibility
    },
}
# Print summary of workload definitions
# Display workload configuration summary
print('Workload Classes:')
for name, cfg in WORKLOADS.items():
    # Show key metrics per workload class
    print(f'  {name}: {cfg["arrival_rate"]} req/s, ctx={cfg["ctx_len"]}, SLO={cfg["ttft_slo_ms"]}ms')

In [ ]:
def simulate_fleet(fleet_size, reserved_frac, workloads, duration, seed):
    """Simulate mixed workload scheduling on a shared GPU fleet.
    
    Returns per-class metrics: SLO attainment, preemption rate,
    utilization over time, and cost breakdown.
    """
    rng = np.random.default_rng(seed)  # reproducible random state
    
    # Split fleet into reserved (P0 only) and shared pools
    reserved_gpus = int(fleet_size * reserved_frac)  # physical isolation for P0
    shared_gpus = fleet_size - reserved_gpus  # priority-scheduled pool
    
    # KV cache memory per token (bytes) for 70B GQA model
    KV_BYTES_PER_TOKEN = 327_680  # 2 * 8 KV heads * 128 dim * 80 layers * 2 bytes
    GPU_HBM_BYTES = 80e9  # A100 80GB
    MODEL_BYTES = 35e9  # 70B INT4 weights
    # Available memory for KV cache after model weights loaded
    AVAILABLE_KV = GPU_HBM_BYTES - MODEL_BYTES - 7e9  # subtract overhead
    
    # Track metrics per workload class
    results = {}
    # Track fleet utilization over time (sample every 10 seconds)
    time_points = np.arange(0, duration, 10)
    utilization_over_time = np.zeros(len(time_points))
    
    for name, cfg in workloads.items():
        # Total requests arriving over simulation duration
        n_requests = int(cfg['arrival_rate'] * duration)
        
        # Memory per request (input + output tokens)
        mem_per_req = (cfg['ctx_len'] + cfg['gen_len']) * KV_BYTES_PER_TOKEN
        # Max concurrent requests per GPU (memory-limited)
        max_concurrent = max(1, int(AVAILABLE_KV / mem_per_req))
        
        # Compute processing time per request (prefill + decode)
        # Prefill: compute-bound, ~50K tokens/sec on H100
        prefill_ms = cfg['ctx_len'] / 50_000 * 1000
        # Decode: memory-bandwidth-bound, ~3ms per token step
        decode_ms = cfg['gen_len'] * 3.0
        # Total GPU time per request
        total_ms = prefill_ms + decode_ms
        
        # Determine effective GPU pool for this workload
        if cfg['priority'] == 0:
            # P0 uses reserved pool exclusively
            effective_gpus = reserved_gpus
        else:
            # Others share the shared pool
            effective_gpus = shared_gpus
        
        # Effective throughput: GPUs * concurrent_per_GPU / time_per_request
        throughput = effective_gpus * max_concurrent / (total_ms / 1000)
        
        # Queue wait time depends on load factor
        load_factor = cfg['arrival_rate'] / throughput  # ratio of demand to capacity
        # M/M/c queue approximation for average wait
        if load_factor < 1.0:
            # System is stable: average wait from queuing theory
            avg_wait_ms = (load_factor / (1 - load_factor)) * total_ms / effective_gpus
        else:
            # System overloaded: requests queue indefinitely
            avg_wait_ms = cfg['ttft_slo_ms'] * 2  # guaranteed SLO violation
        
        # TTFT = queue wait + prefill time
        ttft_samples = rng.exponential(avg_wait_ms + prefill_ms, size=n_requests)
        # SLO attainment: fraction of requests meeting TTFT target
        slo_met = np.mean(ttft_samples < cfg['ttft_slo_ms'])
        
        # Preemption rate: higher priority workloads preempt lower ones
        # P0 never preempted, P3 preempted when shared pool > 80% utilized
        if cfg['priority'] == 0:
            preemption_rate = 0.0  # never preempted (reserved pool)
        elif cfg['priority'] == 1:
            preemption_rate = max(0, (load_factor - 0.9) * 0.1)  # rare
        else:
            preemption_rate = max(0, min(0.2, (load_factor - 0.7) * 0.5))  # frequent
        
        # Cost calculation: GPU-seconds * SLO multiplier
        gpu_seconds = n_requests * total_ms / 1000
        cost = gpu_seconds * cfg['cost_multiplier']
        
        # Store results for this workload class
        results[name] = {
            'n_requests': n_requests,
            'slo_attainment': slo_met,
            'preemption_rate': preemption_rate,
            'avg_ttft_ms': np.mean(ttft_samples),
            'p99_ttft_ms': np.percentile(ttft_samples, 99),
            'cost_gpu_seconds': gpu_seconds,
            'weighted_cost': cost,
            'load_factor': load_factor,
            'max_concurrent_per_gpu': max_concurrent,
        }
    
    # Compute overall fleet utilization
    total_gpu_seconds_used = sum(r['cost_gpu_seconds'] for r in results.values())
    total_gpu_seconds_available = fleet_size * duration
    overall_utilization = total_gpu_seconds_used / total_gpu_seconds_available
    
    return results, overall_utilization

# Run simulation with configured parameters
results, utilization = simulate_fleet(FLEET_SIZE, RESERVED_FRACTION, WORKLOADS, SIM_DURATION, SEED)
print(f'\nFleet utilization: {utilization:.1%}')

In [ ]:
def plot_slo_attainment(results):
    """Bar chart showing SLO attainment per workload class."""
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    names = list(results.keys())  # workload class names for x-axis  # workload class names
    # Color mapping: green for high attainment, red for low
    colors = ['#dcfce7', '#dbeafe', '#fef3c7']  # green, blue, amber per class
    
    # Panel 1: SLO attainment percentage per class
    # Extract SLO attainment percentages for each workload class
    attainments = [results[n]['slo_attainment'] * 100 for n in names]
    axes[0].bar(names, attainments, color=colors, edgecolor='black', linewidth=1.2)
    axes[0].axhline(y=95, color='red', linestyle='--', label='95% target')
    axes[0].set_ylabel('SLO Attainment (%)')  # percentage of requests meeting target
    axes[0].set_title('SLO Attainment by Class')
    axes[0].set_ylim(0, 105)
    axes[0].legend()
    
    # Panel 2: Preemption rate per class
    # Extract preemption rates showing scheduling interference
    preemptions = [results[n]['preemption_rate'] * 100 for n in names]
    axes[1].bar(names, preemptions, color=colors, edgecolor='black', linewidth=1.2)
    axes[1].set_ylabel('Preemption Rate (%)')  # fraction of requests interrupted
    axes[1].set_title('Preemption Rate by Class')
    axes[1].set_ylim(0, 25)
    
    # Panel 3: Cost distribution (weighted by SLO multiplier)
    # Extract SLO-weighted cost allocation per class
    costs = [results[n]['weighted_cost'] for n in names]
    axes[2].bar(names, costs, color=colors, edgecolor='black', linewidth=1.2)
    axes[2].set_ylabel('Weighted Cost (GPU-sec * multiplier)')  # SLO-weighted cost
    axes[2].set_title('Cost Allocation by Class')
    
    plt.tight_layout()
    plt.show()

# Generate the three-panel visualization
plot_slo_attainment(results)

In [ ]:
def plot_preemption_tradeoff():
    """Visualize swap vs recompute cost as a function of context length."""
    # Context lengths from 512 to 32K tokens
    ctx_lengths = np.linspace(512, 32768, 100)
    
    # KV cache size in GB for each context length
    kv_sizes_gb = ctx_lengths * 327_680 / 1e9  # 0.31 MB/token for 70B GQA
    
    # Swap cost: transfer KV to CPU and back via PCIe 4.0 (32 GB/s)
    swap_ms = kv_sizes_gb / 32 * 2 * 1000  # round-trip transfer time
    
    # Recompute cost: re-run prefill at 50K tokens/sec
    recompute_ms = ctx_lengths / 50_000 * 1000  # prefill time in ms
    
    # Find crossover point where swap and recompute costs are equal
    crossover_idx = np.argmin(np.abs(swap_ms - recompute_ms))
    crossover_ctx = ctx_lengths[crossover_idx]  # context length at crossover
    
    fig, ax = plt.subplots(figsize=(10, 5))
    # Plot both cost curves
    ax.plot(ctx_lengths, swap_ms, 'b-', linewidth=2, label='Swap (PCIe 4.0)')
    ax.plot(ctx_lengths, recompute_ms, 'r-', linewidth=2, label='Recompute (prefill)')
    # Mark the crossover point
    ax.axvline(x=crossover_ctx, color='gray', linestyle='--', alpha=0.7)
    ax.annotate(f'Crossover: {crossover_ctx:.0f} tokens',
                xy=(crossover_ctx, swap_ms[crossover_idx]),
                xytext=(crossover_ctx + 3000, swap_ms[crossover_idx] + 50),
                fontsize=10, arrowprops=dict(arrowstyle='->', color='black'))
    
    # Shade regions showing which strategy wins
    ax.fill_between(ctx_lengths[:crossover_idx], 0, swap_ms[:crossover_idx],
                    alpha=0.1, color='red', label='Recompute wins (short ctx)')
    ax.fill_between(ctx_lengths[crossover_idx:], 0, recompute_ms[crossover_idx:],
                    alpha=0.1, color='blue', label='Swap wins (long ctx)')
    
    ax.set_xlabel('Context Length (tokens)')  # x-axis: prompt size
    ax.set_ylabel('Preemption Cost (ms)')  # y-axis: time penalty
    ax.set_title('Preemption Strategy: Swap vs Recompute')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print the decision rule derived from the crossover
    print(f'Decision rule: Swap when context > {crossover_ctx:.0f} tokens, recompute below.')

# Visualize the preemption cost tradeoff
plot_preemption_tradeoff()

In [ ]:
def experiment_fleet_mix():
    """Experiment: How does reserved pool fraction affect P0 SLO and overall utilization?"""
    # Sweep reserved fraction from 0% to 30%
    # Sweep reserved fraction from 0% to 30% in 15 steps
    fractions = np.linspace(0.0, 0.30, 15)
    # Track P0 SLO attainment and fleet utilization at each point
    p0_slos = []  # SLO attainment for real-time workload at each fraction  # SLO attainment for real-time workload
    utils = []  # overall fleet utilization at each fraction  # overall fleet utilization
    
    # Simulate at each reservation level to find optimal tradeoff
    for frac in fractions:
        # Run simulation with this reserved fraction
        r, u = simulate_fleet(FLEET_SIZE, frac, WORKLOADS, SIM_DURATION, SEED)
        p0_slos.append(r['P0_realtime']['slo_attainment'] * 100)
        utils.append(u * 100)
    
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    # Left y-axis: P0 SLO attainment (higher is better)
    color1 = '#2563eb'
    ax1.plot(fractions * 100, p0_slos, 'o-', color=color1, linewidth=2, label='P0 SLO Attainment')
    ax1.set_xlabel('Reserved Pool Fraction (%)')  # x-axis: isolation level
    ax1.set_ylabel('P0 SLO Attainment (%)', color=color1)
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.axhline(y=99.9, color=color1, linestyle=':', alpha=0.5, label='99.9% target')
    
    # Right y-axis: fleet utilization (higher means less waste)
    ax2 = ax1.twinx()
    color2 = '#166534'
    ax2.plot(fractions * 100, utils, 's-', color=color2, linewidth=2, label='Fleet Utilization')
    ax2.set_ylabel('Fleet Utilization (%)', color=color2)
    ax2.tick_params(axis='y', labelcolor=color2)
    
    # Title explaining the tradeoff being visualized
    ax1.set_title('Tradeoff: P0 Isolation vs Fleet Utilization')
    # Combine legends from both axes
    # Combine legends from both y-axes into single legend box
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
    ax1.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Run the fleet mix experiment
experiment_fleet_mix()

In [ ]:
def experiment_cost_savings():
    """Compare cost of dedicated pools vs shared fleet with SLO-weighted pricing."""
    # Dedicated: each workload gets its own pool sized for peak
    # Shared: all workloads on one fleet with priority scheduling
    
    GPU_COST_PER_HOUR = 2.50  # dollars per GPU hour (A100)
    
    # Dedicated pool sizing: each workload needs enough GPUs for its own peak
    dedicated_gpus = {}  # GPUs needed per workload in dedicated mode
    for name, cfg in WORKLOADS.items():
        # Memory per request determines max concurrent per GPU
        mem_per_req = (cfg['ctx_len'] + cfg['gen_len']) * 327_680
        max_conc = max(1, int(38e9 / mem_per_req))  # 38GB KV budget
        # Processing time determines throughput per GPU
        proc_time_s = (cfg['ctx_len'] / 50_000) + (cfg['gen_len'] * 0.003)
        throughput = max_conc / proc_time_s  # requests per second per GPU
        # GPUs needed at peak (with 30% headroom)
        dedicated_gpus[name] = int(np.ceil(cfg['arrival_rate'] / throughput * 1.3))
    
    total_dedicated = sum(dedicated_gpus.values())  # total GPUs if fully dedicated
    # Shared fleet needs fewer GPUs due to statistical multiplexing
    total_shared = int(total_dedicated * 0.6)  # ~40% savings from sharing
    
    # Monthly cost comparison
    hours_per_month = 730  # average hours in a month
    dedicated_cost = total_dedicated * GPU_COST_PER_HOUR * hours_per_month
    shared_cost = total_shared * GPU_COST_PER_HOUR * hours_per_month
    # Instance mix optimization (reserved + spot) saves additional 36%
    optimized_cost = shared_cost * 0.64  # 36% savings from instance mix
    
    # Visualize the three cost scenarios
    fig, ax = plt.subplots(figsize=(8, 5))
    scenarios = ['Dedicated\nPools', 'Shared\nFleet', 'Shared +\nInstance Mix']
    costs = [dedicated_cost / 1000, shared_cost / 1000, optimized_cost / 1000]  # in $K
    colors = ['#ffe4e6', '#dbeafe', '#dcfce7']  # red=expensive, blue=mid, green=optimal
    
    bars = ax.bar(scenarios, costs, color=colors, edgecolor='black', linewidth=1.2)
    ax.set_ylabel('Monthly Cost ($K)')  # thousands of dollars per month
    ax.set_title('Cost Comparison: Dedicated vs Shared Fleet')
    
    # Annotate each bar with the dollar value and GPU count
    for bar, cost, label in zip(bars, costs, [f'{total_dedicated} GPUs', f'{total_shared} GPUs', f'{total_shared} GPUs\n(mixed instances)']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'${cost:.0f}K\n{label}', ha='center', va='bottom', fontsize=9)
    
    # Show savings percentage relative to dedicated
    savings_shared = (1 - shared_cost/dedicated_cost) * 100
    savings_optimized = (1 - optimized_cost/dedicated_cost) * 100
    ax.text(0.95, 0.95, f'Shared saves: {savings_shared:.0f}%\nOptimized saves: {savings_optimized:.0f}%',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='#f3e8ff', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# Run cost comparison experiment
experiment_cost_savings()